# Generación intensiva de mallas

Ejecutar este documento en forma dinámica: [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rirastorza/Intro2FEM/blob/master/Apendices/ejemplo_generacion_sensibilidad_mallas.ipynb)

## Introducción

Este ejemplo puede servir como disparador para generar mallas de manera intensiva y hacer diferentes análisis de sensibilidad. En este caso, utilizaremos una malla simple y evaluaremos el tamaño de elemento y generaremos tres mallas: gruesa, media y fina; y crearemos una carpeta para cada una, de tal manera que los archivos queden organizados.

Esto lo podremos hacer de varias formas, aquí mostramos dos: la primera mediante el uso de la API de python para Gmsh y la segunda utilizando Gmsh (escribiendo el .geo) y python para el scripting.

Lo haremos en una geometría que es una placa 3D que surge de la extrusión de un plano, todo con el CAD nativo de Gmsh (Built-in).

In [ ]:
%%capture
try:
    import gmsh
except ImportError:
    !wget "https://fem-on-colab.github.io/releases/gmsh-install.sh" -O "/tmp/gmsh-install.sh" && bash "/tmp/gmsh-install.sh"
    import gmsh

### 1. Utilizando API de python para Gmsh

Nota: Si tengo todo instalado en mi máquina podré ejecutar el archivo ejemplo_generacion_malla_forma1.py que se encuentra en la carpeta Apendices de este repositorio.

Primero importamos las librerías necesarias. Aquí aparece, además de la API de Gmsh, la librería [os](https://docs.python.org/es/3/library/os.html) (la cual permite controlar el sistema operativo, es decir, es como si ejecutáramos el código por consola). 

In [1]:
import gmsh
import os

Luego definimos los parámetros, noten que podemos definir las resoluciones deseadas en un diccionario cambiando la longitud característica de los elementos. Esto en Gmsh se hace modificando Mesh.CharacteristicLengthMax (más abajo se muestra).

In [2]:
L = 3.5e-3
H = 10.5e-3

#valor de Mesh.CharacteristicLengthMax
resoluciones = {
    "gruesa": 2e-3,
    "media": 1e-3,
    "fina": 5e-4
}

Ahora creamos una función que cree la malla y cuyos argumentos sean justamente, los nombres de las carpetas deseadas y las resoluciones respectivas.


In [9]:
def generar_malla(nombre_carpeta, lc_max):

    gmsh.initialize()
    gmsh.model.add("placa_3d")

    # -------------------------
    # Geometría
    # Se usa lc arbitrario en los puntos
    # porque luego se controla globalmente
    # con CharacteristicLengthMax
    lc = 1.0

    p1 = gmsh.model.geo.addPoint(-L,  L, 0, lc)
    p2 = gmsh.model.geo.addPoint( L,  L, 0, lc)
    p3 = gmsh.model.geo.addPoint( L, -L, 0, lc)
    p4 = gmsh.model.geo.addPoint(-L, -L, 0, lc)

    l1 = gmsh.model.geo.addLine(p1, p2)
    l2 = gmsh.model.geo.addLine(p2, p3)
    l3 = gmsh.model.geo.addLine(p3, p4)
    l4 = gmsh.model.geo.addLine(p4, p1)

    # Superficie
    loop = gmsh.model.geo.addCurveLoop([l1, l2, l3, l4])
    surf = gmsh.model.geo.addPlaneSurface([loop])

    # Extrusión
    ext = gmsh.model.geo.extrude([(2, surf)], 0, 0, H)

    top_surface = ext[0][1]
    volume = ext[1][1]

    lateral_surfaces = [e[1] for e in ext[2:]]

    gmsh.model.geo.synchronize()

    # -------------------------
    # Physical groups

    gmsh.model.addPhysicalGroup(3, [volume], name="interno")

    gmsh.model.addPhysicalGroup(2, [top_surface], name="frente")
    gmsh.model.addPhysicalGroup(2, [surf], name="atras")

    for i, s in enumerate(lateral_surfaces):
        gmsh.model.addPhysicalGroup(2, [s], name=f"lado_{i+1}")

    # -------------------------
    # Control global de tamaño de malla

    gmsh.option.setNumber("Mesh.CharacteristicLengthMax", lc_max)


    # -------------------------
    # Generar malla
    gmsh.model.mesh.generate(3)

    # -------------------------
    # Crear carpeta
    os.makedirs(nombre_carpeta, exist_ok=True)

    # Guardar archivo
    archivo_salida = os.path.join(nombre_carpeta, "placa.msh")
    gmsh.write(archivo_salida)

    print(f"Malla guardada en: {archivo_salida}")

    gmsh.finalize()

Noten que utilizando *os.makedirs* hacemos los directorios en el path donde nos encontremos. Luego, con *os.path.join* tomamos el path de la carpeta creada y guardamos con *gmsh.write* la malla generada con el nombre placa.msh.

Finalmente, generamos las mallas.

In [7]:
for nombre, lc in resoluciones.items():
    generar_malla(nombre, lc)


Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1 (Line)
Info    : [ 10%] Meshing curve 2 (Line)
Info    : [ 20%] Meshing curve 3 (Line)
Info    : [ 30%] Meshing curve 4 (Line)
Info    : [ 40%] Meshing curve 6 (Line)
Info    : [ 50%] Meshing curve 7 (Line)
Info    : [ 50%] Meshing curve 8 (Line)
Info    : [ 60%] Meshing curve 9 (Line)
Info    : [ 70%] Meshing curve 11 (Line)
Info    : [ 80%] Meshing curve 12 (Line)
Info    : [ 90%] Meshing curve 16 (Line)
Info    : [100%] Meshing curve 20 (Line)
Info    : Done meshing 1D (Wall 0.00091168s, CPU 0.001415s)
Info    : Meshing 2D...
Info    : [  0%] Meshing surface 1 (Plane, Frontal-Delaunay)
Info    : [ 20%] Meshing surface 13 (Surface, Frontal-Delaunay)
Info    : [ 40%] Meshing surface 17 (Surface, Frontal-Delaunay)
Info    : [ 50%] Meshing surface 21 (Surface, Frontal-Delaunay)
Info    : [ 70%] Meshing surface 25 (Surface, Frontal-Delaunay)
Info    : [ 90%] Meshing surface 26 (Plane, Frontal-Delaunay)
Info    : Done meshing 2D (Wa

Info    : It. 2000 - 1990 nodes created - worst tet radius 1.03406 (nodes removed 0 10)
Info    : 3D refinement terminated (4091 nodes total):
Info    :  - 3 Delaunay cavities modified for star shapeness
Info    :  - 10 nodes could not be inserted
Info    :  - 19804 tetrahedra created in 0.0683475 sec. (289754 tets/s)
Info    : 0 node relocations
Info    : Done meshing 3D (Wall 0.162769s, CPU 0.164314s)
Info    : Optimizing mesh...
Info    : Optimizing volume 1
Info    : Optimization starts (volume = 5.145e-07) with worst = 0.0124908 / average = 0.780643:
Info    : 0.00 < quality < 0.10 :        55 elements
Info    : 0.10 < quality < 0.20 :       128 elements
Info    : 0.20 < quality < 0.30 :       219 elements
Info    : 0.30 < quality < 0.40 :       302 elements
Info    : 0.40 < quality < 0.50 :       484 elements
Info    : 0.50 < quality < 0.60 :       819 elements
Info    : 0.60 < quality < 0.70 :      2153 elements
Info    : 0.70 < quality < 0.80 :      4647 elements
Info    : 0.80

### 2. Utilizando python para scripting y crear el .geo.

Esta es otra forma un poco más antigua de hacerlo, pero que también funciona si no se quiere utilizar la API de gmsh. Utilizamos *os* nuevamente, pero directamente le pasamos un string para escribir todo el archivo .geo. Esta es la gran diferencia, me queda el .geo en la misma carpeta que se creó. Luego, con *os.system(comando)* se ejecuta el Gmsh en el path respectivo.

Nota: Si tengo todo instalado en mi máquina podré ejecutar el archivo ejemplo_generacion_malla_forma2.py que se encuentra en la carpeta Apendices de este repositorio.

In [10]:
import os

# -------------------------
# Resoluciones de malla
resoluciones = {
    "gruesa": 2e-3,
    "media": 1e-3,
    "fina": 5e-4
}

# -------------------------
# Template del .geo
geo_template = r'''
//Ejemplo 1: Placa 3d
L = 3.5e-3;
H = 10.5e-3;
Point(1) = {-L, L, 0, 1};
Point(2) = {L, L, 0, 1};
Point(3) = {L, -L, 0, 1};
Point(4) = {-L, -L, 0, 1};

Line(1) = {1, 2};
Line(2) = {2, 3};
Line(3) = {3, 4};
Line(4) = {4, 1};

Line Loop(6) = {4, 1, 2, 3};
Plane Surface(6) = {6};

Extrude {0, 0, H} {
 Surface{6};
}

Physical Volume("interno") = {1};

Physical Surface("frente",1) = {28};
Physical Surface("atras",2) = {6};

Physical Surface("abajo",3) = {27};
Physical Surface("izquierda",4) = {15};
Physical Surface("arriba",5) = {19};
Physical Surface("derecha",6) = {23};

Mesh.CharacteristicLengthMax = sizemesh;
'''

# -------------------------
# Generar las tres mallas
for nombre, sizemesh in resoluciones.items():

    # Crear carpeta
    os.makedirs(nombre, exist_ok=True)

    # Reemplazar sizemesh
    geo_content = geo_template.replace(
        "sizemesh",
        f"{sizemesh}"
    )

    # Guardar archivo .geo
    geo_file = os.path.join(nombre, "placa.geo")

    with open(geo_file, "w") as f:
        f.write(geo_content)

    # Archivo de salida
    msh_file = os.path.join(nombre, "placa.msh")

    # -------------------------
    # Ejecutar gmsh por línea de comando

    comando = f"gmsh {geo_file} -3 -o {msh_file}"

    print("Ejecutando:")
    print(comando)

    os.system(comando)

    print(f"Malla generada en: {msh_file}") 

Ejecutando:
gmsh gruesa/placa.geo -3 -o gruesa/placa.msh
Info    : Running 'gmsh gruesa/placa.geo -3 -o gruesa/placa.msh' [Gmsh 4.12.1, 1 node, max. 1 thread]
Info    : Started on Wed May 13 10:20:08 2026
Info    : Reading 'gruesa/placa.geo'...
Info    : Done reading 'gruesa/placa.geo'
Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1 (Line)
Info    : [ 10%] Meshing curve 2 (Line)
Info    : [ 20%] Meshing curve 3 (Line)
Info    : [ 30%] Meshing curve 4 (Line)
Info    : [ 40%] Meshing curve 8 (Line)
Info    : [ 50%] Meshing curve 9 (Line)
Info    : [ 50%] Meshing curve 10 (Line)
Info    : [ 60%] Meshing curve 11 (Line)
Info    : [ 70%] Meshing curve 13 (Line)
Info    : [ 80%] Meshing curve 14 (Line)
Info    : [ 90%] Meshing curve 18 (Line)
Info    : [100%] Meshing curve 22 (Line)
Info    : Done meshing 1D (Wall 0.000508745s, CPU 0.000509s)
Info    : Meshing 2D...
Info    : [  0%] Meshing surface 6 (Plane, Frontal-Delaunay)
Info    : [ 20%] Meshing surface 15 (Surface, Frontal-Del

Info    :  - Creating surface mesh
Info    :  - Identifying boundary edges
Info    :  - Recovering boundary
Info    :  - Added 2 Steiner points
Info    : Done reconstructing mesh (Wall 0.0343374s, CPU 0.034277s)
Info    : Found volume 1
Info    : It. 0 - 0 nodes created - worst tet radius 28.9858 (nodes removed 0 0)
Info    : It. 500 - 488 nodes created - worst tet radius 1.57017 (nodes removed 0 12)
Info    : It. 1000 - 988 nodes created - worst tet radius 1.26893 (nodes removed 0 12)
Info    : It. 1500 - 1488 nodes created - worst tet radius 1.13084 (nodes removed 0 12)
Info    : It. 2000 - 1988 nodes created - worst tet radius 1.03639 (nodes removed 0 12)
Info    : 3D refinement terminated (4091 nodes total):
Info    :  - 3 Delaunay cavities modified for star shapeness
Info    :  - 12 nodes could not be inserted
Info    :  - 19779 tetrahedra created in 0.0674911 sec. (293060 tets/s)
Info    : 0 node relocations
Info    : Done meshing 3D (Wall 0.161852s, CPU 0.16081s)
Info    : Optim